# Argmax Inference with Pretrained Models

This notebook performs semantic correspondence inference using the simple argmax strategy with pretrained models (no fine-tuning).

## What this notebook does:
- Tests all backbone models (DINOv2, DINOv3, SAM) with pretrained weights
- Evaluates on all datasets (SPair-71k, PF-Pascal, PF-Willow)
- Uses simple argmax prediction strategy
- Reports PCK@α metrics for α ∈ {0.05, 0.1, 0.2}

## Configuration options:
- Change `BACKBONE` to 'dinov2', 'dinov3', or 'sam'
- Change `DATASET` to 'spair71k', 'pf-pascal', or 'pf-willow'
- Change `SPLIT` to 'test', 'val', or 'train' (for spair71k)

## Expected runtime:
- ~5-15 minutes per dataset/backbone combination

In [ ]:
%pip install torchmetrics                # ONLY FOR FIRST EXECUTION
%pip install git+https://github.com/facebookresearch/segment-anything.git

from google.colab import drive
import os

REPO_URL = "https://github.com/AML-Semantic-Correspondence/Semantic_Correspondence.git"

# 2. Clone/Pull the Code (access to logic)
print("\n Setting up repository...")

# First ensure we're in a safe directory
%cd /content

# Clean up any existing problematic directories
if os.path.exists('/content/Semantic_Correspondence'):
    print("Removing existing Semantic_Correspondence directory...")
    !rm -rf /content/Semantic_Correspondence

if os.path.exists('/content/semantic-correspondence'):
    print("Removing existing semantic-correspondence directory...")
    !rm -rf /content/semantic-correspondence

# Clone the repository
print("Cloning repository fresh...")
try:
    !git clone {REPO_URL}
    
    # The repo will be cloned as 'Semantic_Correspondence', let's rename it for consistency
    if os.path.exists('/content/Semantic_Correspondence'):
        !mv /content/Semantic_Correspondence /content/semantic-correspondence
        print(" Repository cloned and renamed successfully")
    else:
        print(" Repository clone failed")
except Exception as e:
    print(f" Error during clone: {e}")

# Mount drive and extract datasets
drive.mount("/content/drive", force_remount=True)
!tar -xzf "/content/drive/MyDrive/AML-Semantic-Correspondence/datasets/SPair-71k.tar.gz"
!unzip -o -q "/content/drive/MyDrive/AML-Semantic-Correspondence/datasets/PF-dataset-PASCAL.zip"
!unzip -o -q "/content/drive/MyDrive/AML-Semantic-Correspondence/datasets/PF-dataset.zip"

# Add the repository to path
%cd /content/semantic-correspondence
import sys
sys.path.append('/content/semantic-correspondence')

# Import inference functions
from src.inference.run_evaluation import run_evaluation
from src.inference.argmax import argmax_strategy

# ===== CONFIGURATION =====
# Change these parameters to test different combinations
BACKBONE = 'dinov2'  # Options: 'dinov2', 'dinov3', 'sam'
DATASET = 'spair71k'   # Options: 'spair71k', 'pf-pascal', 'pf-willow'
SPLIT = 'test'         # Options: 'test', 'val', 'train' (only for spair71k)

print(f"\n Running Argmax Inference (Pretrained)")
print(f" Configuration:")
print(f"   - Backbone: {BACKBONE}")
print(f"   - Dataset: {DATASET}")
print(f"   - Split: {SPLIT}")
print(f"   - Strategy: Simple Argmax")
print(f"   - Model: Pretrained (no fine-tuning)")

# Run evaluation
print("\n Starting evaluation...")
run_evaluation(
    backbone=BACKBONE,
    dataset_var=DATASET,
    split=SPLIT,
    prediction_method=argmax_strategy,
    weights_path=None  # Use pretrained weights
)

print("\n Evaluation completed!")
print("\n To test other combinations, modify the BACKBONE, DATASET, and SPLIT variables above and re-run the cell.")